# A_Recuperer — Notebook interactif

Pipeline pour identifier les albums à récupérer :
1. Scan de la bibliothèque locale
2. Matching playlists Spotify vs bibliothèque
3. Scraper Lyon + Qobuz

In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Add service root to path so utils/ is importable
sys.path.insert(0, str(Path().resolve()))

ROOT = Path().resolve().parent.parent
DATA_DIR = ROOT / 'data'
PLAYLISTS_DIR = DATA_DIR / 'Playlists_Spotify'
RESSOURCES_DIR = DATA_DIR / 'Ressources'
BIBLIOTHEQUE_CSV = DATA_DIR / 'Bibliotheque' / 'bibliotheque.csv'
RECHERCHES_XLSX = RESSOURCES_DIR / 'recherches_effectuees.xlsx'
ALBUMS_A_RECHERCHER_CSV = RESSOURCES_DIR / 'albums_a_rechercher.csv'
RESULTATS_CSV = RESSOURCES_DIR / 'resultats_cotes.csv'

## 1. Scan de la bibliothèque locale

In [ ]:
from utils.library import scan_library

LIBRARY_PATH = '/mnt/m/musiques/__Autres'  # Adapter si besoin
df_biblio = scan_library(LIBRARY_PATH, output_path=BIBLIOTHEQUE_CSV)
print(f'{len(df_biblio)} albums dans la bibliothèque')
df_biblio.head()

## 2. Chargement des playlists

In [ ]:
from utils.data_loader import load_playlists, load_recherches_effectuees

df_playlists = load_playlists(PLAYLISTS_DIR)
print(f'{len(df_playlists)} titres dans les playlists')
print(f'Colonnes : {list(df_playlists.columns)}')
df_playlists.head()

## 3. Matching playlists vs bibliothèque

In [ ]:
from utils.matching import clean_albums, clean_artist, match_albums_with_fuzz

ARTIST_THRESHOLD = 90
ALBUM_THRESHOLD = 80

# Préparer les données
df_to_match = df_playlists[['Artist', 'Album']].drop_duplicates().copy()
df_to_match['Artist'] = df_to_match['Artist'].apply(clean_artist)
df_to_match['Album'] = df_to_match['Album'].apply(clean_albums)
df_to_match = df_to_match.drop_duplicates()

df_biblio_clean = df_biblio.copy()
df_biblio_clean['Artist'] = df_biblio_clean['Artist'].apply(clean_artist)
df_biblio_clean['Album'] = df_biblio_clean['Album'].apply(clean_albums)

print(f'Matching {len(df_to_match)} albums...')
df_match = match_albums_with_fuzz(
    df_to_match, df_biblio_clean,
    name_tester='Playlist', name_ressource='Biblio',
    artist_similarity_threshold=ARTIST_THRESHOLD
)
df_match.head()

In [ ]:
# Albums non trouvés en bibliothèque
df_a_recuperer = df_match[df_match['Album_sim'] < ALBUM_THRESHOLD][['Artist_Playlist', 'Album_Playlist']]
df_a_recuperer = df_a_recuperer.rename(columns={'Artist_Playlist': 'Artist', 'Album_Playlist': 'Album'})

# Exclure les déjà recherchés
if RECHERCHES_XLSX.exists():
    df_recherches = load_recherches_effectuees(RECHERCHES_XLSX)
    already_done = set(zip(df_recherches['Artist'], df_recherches['Album']))
    df_a_recuperer = df_a_recuperer[~df_a_recuperer.apply(lambda r: (r['Artist'], r['Album']) in already_done, axis=1)]

df_a_recuperer = df_a_recuperer.drop_duplicates().sort_values(['Artist', 'Album'])
df_a_recuperer.to_csv(ALBUMS_A_RECHERCHER_CSV, index=False)
print(f'{len(df_a_recuperer)} albums à rechercher → {ALBUMS_A_RECHERCHER_CSV}')
df_a_recuperer.head(20)

## 4. Scraper Lyon + Qobuz

In [ ]:
from utils.scraper import run_scraper

# Lance le scraper (peut prendre du temps)
run_scraper(ALBUMS_A_RECHERCHER_CSV, RESULTATS_CSV)

## 5. Résultats

In [ ]:
df_resultats = pd.read_csv(RESULTATS_CSV)
print(f'{len(df_resultats)} résultats')
print(df_resultats['Status'].value_counts())
df_resultats.head(20)